# Week 2 · Day 4 — From pandas to Polars

*The same coffee orders, a faster engine.*

**By the end you'll have shipped:** the exact **sales-summary-by-category** report from Day 1 — rebuilt in **Polars** — plus a **lazy** version that only reads what it needs.

> Core Path = everything unmarked. `Go Deeper 🔧` = optional, for the technical folks.
> This lesson leans on **Week 2 Day 1 (pandas)** the whole way — same data, same four moves.

### 📋 Lesson card

| | |
|---|---|
| **Module** | M1 · Python foundations (Week 2) |
| **Prerequisites** | Week 2 Days 1–3 (pandas) |
| **Est. time** | ~30 min |
| **Capstone tie-in** | *Matter Intelligence* — the same tables, ready to scale to real document volumes |
| **Difficulty** | Core (+ optional Go Deeper) |

### 🎯 Learning objectives

- Say what **Polars** is and *why* it's fast (Rust + Arrow + multithreading + lazy execution).
- Do the four core moves in Polars — **select, filter, sort, group** — and map each to the pandas you already know.
- Understand the **expression API** (`pl.col(...)`), Polars' one big mental-model shift.
- Use **lazy mode** (`scan_csv` → `.collect()`) and explain when it matters.
- Decide, honestly, **when to reach for Polars vs pandas**.

### ⚖️ Why it matters

pandas is the right place to *learn* — it's everywhere and the concepts transfer. But the moment a tool points at **real volumes** — every transaction from a year of stores, millions of rows — speed and memory start to bite. **Polars** does the same select/filter/group work, often **5–15× faster** on large data, and can process files **bigger than your laptop's memory**. Same thinking, a stronger engine. Learning it now means your tools won't hit a wall when the data gets real.

### 🤔 Is Polars really "more powerful" than pandas? (the honest answer)

Mostly **yes on speed and scale**, with nuance:

- **Faster** — written in **Rust**, uses the **Apache Arrow** columnar format, and uses **all your CPU cores** by default (pandas is mostly single-core). On big joins, group-bys, and CSV/Parquet I/O the gap is large.
- **Handles bigger-than-memory data** — its **lazy engine** streams a file in chunks and only computes what you ask for. pandas loads everything into RAM.
- **A cleaner, more consistent API** — one **expression** system for everything, which prevents a lot of pandas foot-guns (the dreaded `SettingWithCopyWarning`, index confusion).

**But pandas still wins on:**

- **Ecosystem & maturity** — far more tutorials, Stack Overflow answers, and libraries that expect a pandas DataFrame. For a team learning to code, that support matters.
- **Tiny data** — for a few hundred orders, both are instant; Polars' speed edge is invisible at that size.

**Bottom line for this team:** learn the concepts in **pandas** (Days 1–3), reach for **Polars** when data gets big or a job feels slow. They interoperate freely, so it's not either/or — you'll use both.

### ⚙️ Setup

Installs Polars if it's missing (guarded, so re-running is safe), imports it, and makes sure the sample `coffee_orders.csv` is reachable — same data as the pandas lesson, so the comparison is apples-to-apples.

In [1]:
import os, sys, subprocess

# guarded install — only runs the first time, safe to re-run
try:
    import polars as pl
    import pyarrow            # fast, zero-copy pandas <-> polars conversion
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "polars", "pyarrow"], check=True)
    import polars as pl

import pandas as pd   # for a few side-by-side comparisons

# reuse the same coffee sample as Week 2 Day 1
SAMPLE = [
    {"order_id":"O-5001","item":"Latte","size":"M","category":"Espresso Drink","price":4.75,"payment":"Card","store":"Downtown"},
    {"order_id":"O-5002","item":"Cold Brew","size":"L","category":"Cold","price":5.35,"payment":"App","store":"Airport"},
    {"order_id":"O-5003","item":"Drip","size":"S","category":"Brewed","price":2.50,"payment":"Cash","store":"Uptown"},
    {"order_id":"O-5004","item":"Mocha","size":"L","category":"Espresso Drink","price":5.95,"payment":"Card","store":"Downtown"},
    {"order_id":"O-5005","item":"Croissant","size":"","category":"Food","price":3.25,"payment":"App","store":"Airport"},
    {"order_id":"O-5006","item":"Cappuccino","size":"M","category":"Espresso Drink","price":4.50,"payment":"Card","store":"Uptown"},
    {"order_id":"O-5007","item":"Latte","size":"L","category":"Espresso Drink","price":5.50,"payment":"Cash","store":"Downtown"},
    {"order_id":"O-5008","item":"Muffin","size":"","category":"Food","price":2.95,"payment":"Card","store":"Airport"},
    {"order_id":"O-5009","item":"Cold Brew","size":"M","category":"Cold","price":4.65,"payment":"App","store":"Uptown"},
    {"order_id":"O-5010","item":"Espresso","size":"S","category":"Espresso Drink","price":2.75,"payment":"Cash","store":"Downtown"},
]

CSV_PATH = "coffee_orders.csv"
SHARED = os.path.join("..", "..", "data", "coffee_orders.csv")   # Training/data/coffee_orders.csv
if os.path.exists(SHARED):
    CSV_PATH = SHARED
elif not os.path.exists(CSV_PATH):
    pl.DataFrame(SAMPLE).write_csv(CSV_PATH)

print(f"polars {pl.__version__} | pandas {pd.__version__} | using CSV: {CSV_PATH}")

polars 1.42.1 | pandas 3.0.3 | using CSV: ../../data/coffee_orders.csv


### 1 · Read a CSV — familiar territory

`pl.read_csv` mirrors `pd.read_csv`. A Polars DataFrame prints with its **column types in the header** — a small but handy touch.

In [2]:
df = pl.read_csv(CSV_PATH)
print(f"shape (rows, cols): {df.shape}")
df.head()

shape (rows, cols): (36, 8)


order_id,date,item,size,category,price,payment,store
str,str,str,str,str,f64,str,str
"""O-5001""","""2026-03-07""","""Cappuccino""","""S""","""Espresso Drink""",3.75,"""Cash""","""Downtown"""
"""O-5002""","""2026-03-07""","""Mocha""","""S""","""Espresso Drink""",4.5,"""Card""","""Airport"""
"""O-5003""","""2026-03-05""","""Cappuccino""","""L""","""Espresso Drink""",5.25,"""Card""","""Downtown"""
"""O-5004""","""2026-03-03""","""Cappuccino""","""S""","""Espresso Drink""",3.75,"""App""","""Airport"""
"""O-5005""","""2026-03-03""","""Latte""","""L""","""Espresso Drink""",5.5,"""App""","""Airport"""


**What just happened:** same table, Polars engine. Notice the dtype shown under each column name (`str`, `f64`) — Polars is explicit about types.

### 2 · The one big idea: **expressions** (`pl.col`)

This is the mental shift. In pandas you write `df["price"] > 5`. In Polars you describe *what you want* with an **expression** — `pl.col("price") > 5` — and hand it to a method like `.filter()` or `.select()`.

Analogy: pandas is you reaching over the counter and grabbing the cups yourself; a Polars **expression** is an **order you call out to the barista** ("every drink over \$5"). Because it's an instruction, Polars can read all your orders together, optimize, and run them in parallel.

In [ ]:
# an expression is just a description — it doesn't run until given to a method
premium = pl.col("price") > 5
print(type(premium))
print(premium)

<class 'polars.expr.expr.Expr'>
[(col("price")) > (dyn int: 5)]


<Expr ['[(col("price")) > (dyn int: 5)…'] at 0x1186A7A50>

### 3 · Select columns  →  like pandas `df[[...]]` / SQL `SELECT`

Use `.select()` with column names or expressions.

In [4]:
df.select(["item", "category", "price"]).head()

item,category,price
str,str,f64
"""Cappuccino""","""Espresso Drink""",3.75
"""Mocha""","""Espresso Drink""",4.5
"""Cappuccino""","""Espresso Drink""",5.25
"""Cappuccino""","""Espresso Drink""",3.75
"""Latte""","""Espresso Drink""",5.5


### 4 · Filter rows  →  like pandas `df[df.x > n]` / SQL `WHERE`

Use `.filter()` with an expression. Combine conditions with `&` / `|` (each wrapped in parentheses), just like pandas.

In [7]:
# premium orders
df.filter(pl.col("price") > 5).select(["order_id", "item", "price"])

order_id,item,price
str,str,f64
"""O-5003""","""Cappuccino""",5.25
"""O-5005""","""Latte""",5.5
"""O-5009""","""Mocha""",5.25
"""O-5014""","""Latte""",5.5
"""O-5017""","""Cold Brew""",5.35
…,…,…
"""O-5025""","""Cold Brew""",5.35
"""O-5030""","""Cold Brew""",5.35
"""O-5031""","""Cappuccino""",5.25


In [8]:
# Downtown AND premium  (pandas: df[(df.store=='Downtown') & (df.price>5)])
df.filter(
    (pl.col("store") == "Downtown") & (pl.col("price") > 5)
).select(["order_id", "item", "store", "price"])

order_id,item,store,price
str,str,str,f64
"""O-5003""","""Cappuccino""","""Downtown""",5.25
"""O-5014""","""Latte""","""Downtown""",5.5
"""O-5017""","""Cold Brew""","""Downtown""",5.35
"""O-5020""","""Mocha""","""Downtown""",5.95
"""O-5022""","""Mocha""","""Downtown""",5.95


### 5 · Sort  →  like pandas `sort_values` / SQL `ORDER BY`

In [9]:
df.sort("price", descending=True).select(["order_id", "item", "price"]).head()

order_id,item,price
str,str,f64
"""O-5020""","""Mocha""",5.95
"""O-5022""","""Mocha""",5.95
"""O-5005""","""Latte""",5.5
"""O-5014""","""Latte""",5.5
"""O-5017""","""Cold Brew""",5.35


### 6 · Group & aggregate  →  like pandas `groupby` / SQL `GROUP BY`

`group_by(...).agg(...)`. Inside `.agg()` you pass expressions describing each summary column. It reads almost like a sentence: *group by category, and for each give me the sum and mean of price.*

In [10]:
by_cat = (
    df.group_by("category")
      .agg([
          pl.col("price").sum().round(2).alias("revenue"),
          pl.col("price").mean().round(2).alias("avg_price"),
          pl.len().alias("orders"),
      ])
      .sort("revenue", descending=True)
)
by_cat

category,revenue,avg_price,orders
str,f64,f64,u32
"""Espresso Drink""",93.9,4.7,20
"""Cold""",29.3,4.88,6
"""Food""",21.55,3.08,7
"""Brewed""",8.45,2.82,3


### 7 · pandas ↔ Polars cheat-sheet

You already know the left column. The right column is today.

| Goal | pandas | Polars | (SQL) |
|---|---|---|---|
| read CSV | `pd.read_csv(p)` | `pl.read_csv(p)` | — |
| pick columns | `df[["a","b"]]` | `df.select(["a","b"])` | `SELECT a, b` |
| filter rows | `df[df.x > 5]` | `df.filter(pl.col("x") > 5)` | `WHERE x > 5` |
| sort | `df.sort_values("x")` | `df.sort("x")` | `ORDER BY x` |
| group + sum | `df.groupby("g")["x"].sum()` | `df.group_by("g").agg(pl.col("x").sum())` | `GROUP BY g` |
| new column | `df["y"] = df.x * 2` | `df.with_columns((pl.col("x")*2).alias("y"))` | `SELECT x*2 AS y` |

The shapes rhyme. The main change is *expressions* (`pl.col`) and method chaining instead of bracket indexing.

### 8 · The superpower: **lazy** mode

Everything above ran **eagerly** — each line executed immediately, like pandas. Polars' real edge is **lazy** mode: you describe the *whole* pipeline first, Polars **optimizes** it (e.g. only reads the columns/rows you actually use), then runs it all at once when you call `.collect()`.

Analogy: instead of walking to the counter for each item, you hand the barista the **entire** order up front — they plan the most efficient route and make **one** trip. On big files this is the difference between minutes and seconds, and it's what lets Polars handle data **larger than memory**.

In [11]:
# scan_csv (not read_csv) starts a LAZY pipeline — nothing is read yet
lazy_summary = (
    pl.scan_csv(CSV_PATH)                                   # a plan, not data
      .filter(pl.col("category") != "Food")
      .group_by("category")
      .agg(pl.col("price").sum().round(2).alias("revenue"))
      .sort("revenue", descending=True)
)

print(type(lazy_summary))          # a LazyFrame — the recipe
result = lazy_summary.collect()    # NOW it runs, optimized, in one pass
result

<class 'polars.lazyframe.frame.LazyFrame'>


category,revenue
str,f64
"""Espresso Drink""",93.9
"""Cold""",29.3
"""Brewed""",8.45


> **`Go Deeper 🔧` — see the optimizer think.** `.explain()` prints the query plan Polars will run. On a real file you'd see it push the filter down into the CSV scan so it never even loads rows it will throw away ("predicate pushdown") and read only needed columns ("projection pushdown").

In [12]:
print(lazy_summary.explain())

SORT BY [descending: [true]] [col("revenue")]
  AGGREGATE[maintain_order: false]
    [col("price").sum().round().alias("revenue")] BY [col("category")]
    FROM
    Csv SCAN [../../data/coffee_orders.csv]
    PROJECT 2/8 COLUMNS
    SELECTION: [(col("category")) != ("Food")]
    ESTIMATED ROWS: 32


> **`Common pitfalls ⚠️` (coming from pandas)**
>
> - **No index.** Polars has no row index — there's no `.loc`/`.iloc` juggling. Filter by condition instead.
> - **Columns via `pl.col("x")`**, not bare `df.x` inside operations.
> - **`group_by`** (underscore) and **`descending=`**, vs pandas' `groupby` and `ascending=`.
> - **Assign with `.with_columns(...)`**, not `df["new"] = ...`. Polars methods return a *new* frame (no in-place surprises).

### 🔁 Interop: they play nicely together

You rarely have to choose globally. Convert in one line — do heavy lifting in Polars, then hand a pandas DataFrame to a library that expects one (like most plotting tools).

In [13]:
pdf = df.to_pandas()          # Polars -> pandas
back = pl.from_pandas(pdf)    # pandas -> Polars
print(f"{type(pdf)} -> {type(back)}")

<class 'pandas.DataFrame'> -> <class 'polars.dataframe.frame.DataFrame'>


### ✍️ Your turn

In [ ]:
df = pl.read_csv(CSV_PATH)

# TODO 1: filter to only 'Espresso Drink' orders, then select order_id, item, price
# TODO 2: compute the AVERAGE price per category (hint: pl.col(...).mean())
# TODO 3: count orders per store (hint: group_by then pl.len())
# TODO 4 (stretch): rewrite TODO 2 as a LAZY pipeline using pl.scan_csv(...).collect()

# your code here


<details><summary>✅ Show solution</summary>

```python
# 1
print(df.filter(pl.col("category") == "Espresso Drink").select(["order_id", "item", "price"]))

# 2
print(df.group_by("category").agg(pl.col("price").mean().round(2).alias("avg_price")))

# 3
print(df.group_by("store").agg(pl.len().alias("orders")))

# 4 (lazy)
print(
    pl.scan_csv(CSV_PATH)
      .group_by("category")
      .agg(pl.col("price").mean().round(2).alias("avg_price"))
      .collect()
)
```
</details>

### 🚀 Build the artifact — the sales summary, in Polars (eager + lazy)

Same deliverable as Day 1 — **read → filter → group → save** — now in Polars, plus a lazy version that would scale to a massive file unchanged.

In [ ]:
# --- eager version ---
df = pl.read_csv(CSV_PATH)
summary = (
    df.filter(pl.col("category") != "Food")
      .group_by("category")
      .agg([
          pl.len().alias("orders"),
          pl.col("price").sum().round(2).alias("total_revenue"),
          pl.col("price").mean().round(2).alias("avg_price"),
      ])
      .sort("total_revenue", descending=True)
)
print(summary)
summary.write_csv("sales_summary_by_category_polars.csv")

# --- same thing, lazy (this pattern is what scales to millions of rows) ---
lazy_result = (
    pl.scan_csv(CSV_PATH)
      .filter(pl.col("category") != "Food")
      .group_by("category")
      .agg(pl.col("price").sum().round(2).alias("total_revenue"))
      .sort("total_revenue", descending=True)
      .collect()
)
print(f"\nLazy result matches: {lazy_result.shape}")
print("\n✅ Shipped: sales_summary_by_category_polars.csv (eager) + a lazy pipeline that scales.")

> **🔗 Your world — from coffee to matters.** This is Day 1's billing report at scale. Point `pl.scan_csv` at a multi-gigabyte `matters.csv`, swap `category` → `practice_area` and `price` → `amount_billed`, and the *same* lazy pipeline streams it in one optimized pass — no code shape change, no out-of-memory crash. The coffee shop taught the moves; Polars runs them at firm scale.

### 📝 Recap — what you shipped

- **Polars** does the same select/filter/sort/group work as pandas, usually **much faster** (Rust + Arrow + multithreading).
- The key shift is the **expression API** (`pl.col(...)`) handed to methods like `.select()` / `.filter()` / `.agg()`.
- **Lazy mode** (`scan_csv` → `.collect()`) optimizes the whole pipeline and handles **bigger-than-memory** data.
- pandas and Polars **interoperate** — convert in one line; use each where it's strongest.
- **Artifact:** the sales summary rebuilt in Polars, eager and lazy.

### 🧠 Check your understanding

1. What are the three main reasons Polars is faster than pandas?
2. What does `pl.col("price")` represent, and how is it different from pandas' `df["price"]`?
3. What's the difference between `pl.read_csv` and `pl.scan_csv`?
4. For a 300-row coffee table, will your team *notice* Polars being faster? Why or why not?

<details><summary>Answers</summary>

1. It's compiled **Rust**, uses the columnar **Arrow** format, and runs **multithreaded** across all CPU cores (plus lazy query optimization).
2. It's an **expression** — a description of a column/operation you hand to a method — so Polars can optimize and parallelize. pandas' `df["..."]` immediately returns the actual column (a Series).
3. `read_csv` loads the file **now** (eager); `scan_csv` starts a **lazy** plan that only runs — optimized — when you call `.collect()`.
4. **No** — at a few hundred rows both are effectively instant. Polars' advantage shows up on **large** data. Learn concepts in pandas; reach for Polars when volume grows.
</details>

### ➡️ Where this goes next

That wraps **Week 2 — pandas & Polars**. You can now load, clean, group, join, and scale tabular data. Next in the foundations track:

- **SQL for Snowflake** — the same select/filter/sort/group verbs as real SQL (with a SQLite fallback). You've now seen this pattern in **three** dialects — pandas, Polars, and soon SQL — which is exactly the point.
- **Building with Claude** — feed a filtered frame (pandas *or* Polars) into an LLM to summarize or classify it (the *Matter Intelligence* capstone).

### 📖 Reference & glossary

| Term | Plain meaning |
|---|---|
| Polars | a fast DataFrame library written in Rust |
| expression (`pl.col`) | a description of a column/operation, handed to a method |
| eager (`read_csv`) | runs each step immediately, like pandas |
| lazy (`scan_csv` + `collect`) | builds & optimizes the whole pipeline, then runs once |
| Apache Arrow | the columnar in-memory format Polars uses for speed |
| `with_columns` | add/replace columns (Polars' assignment) |

**Official docs:** [Polars user guide](https://docs.pola.rs/) · [Coming from pandas](https://docs.pola.rs/user-guide/migration/pandas/) · [Lazy API](https://docs.pola.rs/user-guide/lazy/)